# supervised variant prediction for multiple genes

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import scipy.stats
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

In [ ]:
import torch
import esm
import pandas as pd
import gc
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set your local ESM model path
MODEL_PATH = "/datasets/bio/esm/models/esm2_t6_8M_UR50D.pt"

def identify_mutations(seq, ref_seq):
    """
    Identifies mutations by comparing the sequence to the reference wild-type protein.
    """
    mutations = []
    min_length = min(len(seq), len(ref_seq))  # Handle different lengths safely

    for i in range(min_length):
        if seq[i] != ref_seq[i]:  # Mutation detected
            mutations.append(f"p.{ref_seq[i]}{i+1}{seq[i]}")  # Format: p.A23T

    return mutations

def generate_embeddings(df, ref_protein, model_path=MODEL_PATH, include=["mean"], truncate_len=1022):
    """
    Extracts embeddings and dynamically identifies mutations without re-downloading the model.

    Args:
        df (pd.DataFrame): DataFrame containing sequences.
        ref_protein (str): Wild-type reference protein sequence.
        model_path (str): Path to locally stored ESM model.
        include (list): List of representations to extract. Options: ["mean", "per_tok", "bos"].
        truncate_len (int): Maximum sequence length before truncation.

    Returns:
        pd.DataFrame: DataFrame containing embeddings and mutations.
    """
    # Load Model from local path
    model, alphabet = esm.pretrained.load_model_and_alphabet_local(model_path)
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Initialize batch converter
    batch_converter = alphabet.get_batch_converter()

    # Prepare storage lists
    embeddings_list = []
    per_token_representations = []
    bos_representations = []
    labels = []
    mutations_all = []

    # Process sequences in batches
    data = list(zip(df["Filename"], df["Protein_Sequence"]))  # Assuming `Filename` is the identifier
    dataloader = DataLoader(data, batch_size=1, shuffle=False)

    with torch.no_grad():
        for labels_batch, sequences_batch in tqdm(dataloader, desc="Processing Batches"):
            batch_labels, batch_strs, batch_tokens = batch_converter(list(zip(labels_batch, sequences_batch)))
            batch_tokens = batch_tokens.to(device)

            # Forward pass
            model_results = model(batch_tokens, repr_layers=[model.num_layers], return_contacts=False)
            token_representations = model_results["representations"][model.num_layers].to("cpu")

            for i, seq in enumerate(batch_strs):
                seq_len = min(truncate_len, len(seq))

                # Mean-pooled embedding
                if "mean" in include:
                    mean_repr = token_representations[i, 1:seq_len+1].mean(0).clone()
                    embeddings_list.append(mean_repr)

                # Per-token representations
                if "per_tok" in include:
                    per_tok_repr = token_representations[i, 1:seq_len+1].clone()
                    per_token_representations.append(per_tok_repr)

                # BOS-token representation
                if "bos" in include:
                    bos_repr = token_representations[i, 0].clone()
                    bos_representations.append(bos_repr)

                # Track labels and mutations
                labels.append(batch_labels[i])

                # **Identify mutations dynamically**
                mutations = identify_mutations(seq, ref_protein)
                mutations_all.append(mutations)
            torch.cuda.empty_cache()
            gc.collect()

    # Convert embeddings to DataFrame
    df_embeddings = pd.DataFrame({
        "identifier": labels,
        "mean_embedding": embeddings_list if "mean" in include else None,
        "per_token_embedding": per_token_representations if "per_tok" in include else None,
        "bos_embedding": bos_representations if "bos" in include else None,
        "mutation": mutations_all  # **Now dynamically computed!**
    })

    # Clean up memory
    gc.collect()
    torch.cuda.empty_cache()

    return df_embeddings


## data generation

In [ ]:
gene_name='inhA'

In [ ]:
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene_name, 'protein_sequence'].values

In [ ]:
gene_sequences=pd.read_csv(f"./data/sequence_data_csv/{gene_name}_ethionamide_sequence_data.csv")

In [ ]:
gene_sequences_filtered = gene_sequences[gene_sequences["Frameshift_Mutation"] == 0].copy()

In [ ]:
# Ensure required columns exist
if "Protein_Sequence" not in gene_sequences_filtered.columns:
    raise ValueError("CSV file must contain a 'Protein_Sequence' column.")

In [ ]:
valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")  # Standard 20 amino acids

# Check each sequence
for idx, seq in enumerate(gene_sequences_filtered["Protein_Sequence"]):
    invalid_chars = [c for c in seq if c not in valid_amino_acids]
    if invalid_chars:
        print(f"WARNING: Sequence {idx} contains invalid characters: {invalid_chars}")


In [ ]:
chunk_size = 1000  # Adjust based on memory availability
for i in range(0, len(gene_sequences_filtered), chunk_size):
    df_chunk = gene_sequences_filtered.iloc[i:i+chunk_size]
    df_embeddings = generate_embeddings(df_chunk, ref_protein, include=["mean"])
    
    # Save or append results immediately to avoid memory buildup
    df_embeddings.to_csv(f"{gene_name}_embeddings_chunk_{i}.csv", index=False)
    # Clean up memory after each chunk!
    del df_embeddings
    torch.cuda.empty_cache()
    gc.collect()



In [ ]:
#combine all embedding files

import pandas as pd
import glob

# Find all embedding CSV files (update the pattern if needed)
embedding_files = glob.glob(f"{gene_name}_embeddings_chunk_*.csv")  
# Load and combine all CSVs
dfs = [pd.read_csv(f) for f in embedding_files]
combined_df = pd.concat(dfs, ignore_index=True)

# Save the combined embeddings file
combined_df.to_csv(f"{gene_name}_ethionamide_all_embeddings.csv", index=False)

print(f" Combined {len(embedding_files)} embedding files into 'all_embeddings.csv'.")
print("Shape of combined embeddings:", combined_df.shape)


In [ ]:
# merge with phenotype
import pandas as pd
import torch

# Load embeddings CSV
embeddings_df = pd.read_csv(f"{gene_name}_ethionamide_all_embeddings.csv")  # Update with actual path


# Load sequences CSV (which contains phenotypes)
sequences_df=pd.read_csv(f"./data/sequence_data_csv/{gene_name}_ethionamide_sequence_data.csv")
sequences_df = sequences_df[sequences_df["Frameshift_Mutation"] == 0].copy()

In [ ]:
# Merge on Filename (sequences) and identifier (embeddings)
merged_df = embeddings_df.merge(sequences_df[['Filename', 'Phenotype']], 
                                left_on="identifier", right_on="Filename", 
                                how="inner")

# Drop redundant Filename column after merging
merged_df.drop(columns=["Filename"], inplace=True)

In [ ]:
# Save the merged dataset
merged_df.to_csv(f"{gene_name}_ethionamide_merged_embeddings_with_phenotype.csv", index=False)

print(f"Merged embeddings with phenotype. Final shape: {merged_df.shape}")

## train model

In [ ]:
import pandas as pd
import numpy as np
import ast
import torch
import scipy.stats
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

# Directory to save results
RESULTS_DIR = "gene_results"
os.makedirs(RESULTS_DIR, exist_ok=True)


# Step 1: Convert String Embeddings to Numerical Arrays
def parse_tensor(tensor_str):
    """Convert string tensor representation into a NumPy array."""
    try:
        return np.array(ast.literal_eval(tensor_str.replace("tensor(", "").replace(")", "")))
    except Exception as e:
        print(f"Error parsing tensor: {tensor_str} -> {e}")
        return np.zeros(320)  # Adjust size if needed


#  Step 2: Load and Merge Data
def load_embeddings(df):
    """
    Load embeddings from a DataFrame and convert to numerical arrays.
    """
    df["mean_embedding"] = df["mean_embedding"].apply(parse_tensor)
    return df



#  Step 3: Extract Features (X) and Target (y)
def prepare_data(df):
    """
    Prepare feature matrix (X) and target labels (y) for training.
    Filters for binary classification: only 'R' and 'S' phenotypes.
    """
    # Filter to only 'R' and 'S'
    df_binary = df[df["Phenotype"].isin(["R", "S"])].copy()

    # Encode labels (R -> 1, S -> 0)
    y = df_binary["Phenotype"].astype("category").cat.codes
    X = np.vstack(df_binary["combined_embedding"].values)

    print(f"Binary classification only. Data prepared. Shape X: {X.shape}, y: {y.shape}, Classes: {np.unique(y)}")
    return X, y



# Step 4: Train-Test Split
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


# Step 5: PCA Visualization and Save
def visualize_pca(X_train, y_train, gene_name, num_components=60):
    """
    Perform PCA and save the visualization.
    """
    pca = PCA(n_components=num_components)
    X_train_pca = pca.fit_transform(X_train)

    plt.figure(figsize=(7, 6))
    sc = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, marker='.')
    plt.xlabel("PCA First Principal Component")
    plt.ylabel("PCA Second Principal Component")
    plt.colorbar(sc, label="Variant Effect")
    plt.title(f"PCA Visualization for {gene_name}")

    # Save PCA Figure
    pca_path = os.path.join(RESULTS_DIR, f"{gene_name}_pca.png")
    plt.savefig(pca_path)
    plt.close()
    
    return pca


# Step 6: Hyperparameter Tuning using GridSearchCV
def run_grid_search(X_train, y_train, num_pca_components=60):
    """
    Perform hyperparameter tuning using GridSearchCV on KNN, SVM, and Random Forest.
    """
    knn_grid = [
        {
            'model': [KNeighborsRegressor()],
            'model__n_neighbors': [5, 10],
            'model__weights': ['uniform', 'distance'],
            'model__algorithm': ['ball_tree', 'kd_tree', 'brute'],
            'model__leaf_size': [15, 30],
            'model__p': [1, 2],
        }
    ]

    svm_grid = [
        {
            'model': [SVR()],
            'model__C': [0.1, 1.0, 10.0],
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__degree': [3],
            'model__gamma': ['scale'],
        }
    ]

    rfr_grid = [
        {
            'model': [RandomForestRegressor()],
            'model__n_estimators': [20],
            'model__criterion': ['squared_error', 'absolute_error'],
            'model__max_features': ['sqrt', 'log2'],
            'model__min_samples_split': [5, 10],
            'model__min_samples_leaf': [1, 4]
        }
    ]

    cls_list = [KNeighborsRegressor, SVR, RandomForestRegressor]
    param_grid_list = [knn_grid, svm_grid, rfr_grid]

    pipe = Pipeline(
        steps=[('pca', PCA(num_pca_components)), ('model', 'passthrough')]
    )

    result_list = []
    grid_list = []

    for cls_name, param_grid in zip(cls_list, param_grid_list):
        print(f"🔹 Running Grid Search for {cls_name.__name__}...")

        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring='r2',
            verbose=1,
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        result_list.append(pd.DataFrame.from_dict(grid.cv_results_))
        grid_list.append(grid)

    return result_list, grid_list


#  Step 7: Evaluate Best Models and Save Results
def evaluate_models_spearman(grid_list, X_test, y_test, gene_name):
    """
    Evaluate the best models using Spearman correlation and save results.
    """
    results = []
    
    for grid in grid_list:
        best_model = grid.best_estimator_.get_params()["steps"][1][1]
        preds = grid.predict(X_test)
        spearman_corr, _ = scipy.stats.spearmanr(y_test, preds)

        # Save results
        results.append({
            "Gene": gene_name,
            "Best Model": str(best_model),
            "Spearman Correlation": spearman_corr
        })

        print(f" Best Model for {gene_name}: {best_model}")
        print(f"Spearman Correlation: {spearman_corr:.4f}")
        print("\n", "-" * 80, "\n")

    return results


from sklearn.metrics import roc_auc_score

#  Step 7: Evaluate Best Models using AUC and Save Results
def evaluate_models(grid_list, X_test, y_test, gene_name):
    """
    Evaluate the best models using AUC and save results.
    """
    results = []

    for grid in grid_list:
        best_model = grid.best_estimator_.get_params()["steps"][1][1]
        preds = grid.predict(X_test)

        try:
            auc = roc_auc_score(y_test, preds)
        except ValueError as e:
            print(f"[ Warning] AUC could not be computed for {gene_name}: {e}")
            auc = None

        results.append({
            "Gene": gene_name,
            "Best Model": str(best_model),
            "AUC": auc
        })

        print(f" Best Model for {gene_name}: {best_model}")
        if auc is not None:
            print(f"AUC: {auc:.4f}")
        else:
            print("AUC: N/A (only one class present in test set)")
        print("\n", "-" * 80, "\n")

    return results



In [ ]:
import pandas as pd

# Load your gene CSVs
df_ethA = pd.read_csv('/data/embeddings/ethA_merged_embeddings_with_phenotype.csv')  # or sep=',' depending on file
df_inhA = pd.read_csv('/data/embeddings/inhA_ethionamide_merged_embeddings_with_phenotype.csv')


In [ ]:
print(df_ethA.shape, df_inhA.shape)

In [ ]:

# Merge on isolate identifier
df_merged = df_ethA[['identifier', 'Phenotype']].merge(
    df_inhA[['identifier', 'Phenotype']], on='identifier', suffixes=('_ethA', '_inhA')
)

# Compare the phenotypes
df_merged['same_phenotype'] = df_merged['Phenotype_ethA'] == df_merged['Phenotype_inhA']

# Check if any discrepancies exist
discrepant = df_merged[~df_merged['same_phenotype']]
print(f"Total discrepancies: {len(discrepant)}")


In [ ]:
# Merge on isolate identifier
df_merged = df_ethA[['identifier', 'Phenotype']].merge(
    df_inhA[['identifier', 'Phenotype']], on='identifier', suffixes=('_ethA', '_inhA')
)

# Identify missing values
df_merged['missing_ethA'] = df_merged['Phenotype_ethA'].isna() | (df_merged['Phenotype_ethA'].astype(str).str.strip() == '')
df_merged['missing_inhA'] = df_merged['Phenotype_inhA'].isna() | (df_merged['Phenotype_inhA'].astype(str).str.strip() == '')

# Compare phenotypes (only if both are present)
df_merged['same_phenotype'] = (
    ~df_merged['missing_ethA'] &
    ~df_merged['missing_inhA'] &
    (df_merged['Phenotype_ethA'] == df_merged['Phenotype_inhA'])
)

# Discrepant rows
discrepant = df_merged[~df_merged['same_phenotype']]
print(f"Total discrepancies (including missing): {len(discrepant)}")

# Optional: Separate true mismatches vs. missing
missing_phenotypes = df_merged[df_merged['missing_ethA'] | df_merged['missing_inhA']]
true_mismatches = discrepant[~discrepant.index.isin(missing_phenotypes.index)]

print(f"Of which:")
print(f"  - Missing phenotype values: {len(missing_phenotypes)}")
print(f"  - True mismatches (non-missing but unequal): {len(true_mismatches)}")


In [ ]:
# Show a few true mismatches
print("Sample true mismatches (both phenotypes present but different):")
print(true_mismatches[['identifier', 'Phenotype_ethA', 'Phenotype_inhA']].head(10))


In [ ]:
df_ethA["mean_embedding_parsed"] = df_ethA["mean_embedding"].apply(parse_tensor)
df_inhA["mean_embedding_parsed"] = df_inhA["mean_embedding"].apply(parse_tensor)

In [ ]:
# merge by identifier
df_merged = df_ethA[["identifier", "mean_embedding_parsed", "Phenotype"]].merge(
    df_inhA[["identifier", "mean_embedding_parsed"]],
    on="identifier",
    suffixes=("_ethA", "_inhA"))

In [ ]:
# concatenate the embedding
df_merged["combined_embedding"] = df_merged.apply(lambda row: np.concatenate([row["mean_embedding_parsed_ethA"], row["mean_embedding_parsed_inhA"]]),axis=1)


In [ ]:
"""
Run the pipeline for each gene and store results in CSV.
"""
all_results = []

gene_name="ethA_inhA"
print(f"Processing gene: {gene_name} ({len(df_merged)} samples)")
X, y = prepare_data(df_merged)
X_train, X_test, y_train, y_test = split_data(X, y)
print(y_test)

# PCA and save visualization
visualize_pca(X_train, y_train, gene_name)

# Hyperparameter tuning
_, grid_list = run_grid_search(X_train, y_train)

# Evaluate and collect results
results = evaluate_models(grid_list, X_test, y_test, gene_name)
all_results.extend(results)

# Save all results to CSV
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(RESULTS_DIR, f"{gene_name}_model_performance.csv"), index=False)




In [ ]:
print(f"\n All results saved in '{RESULTS_DIR}/model_performance.csv'")

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Define the directory containing the CSV files
directory_path = "./esm/gene_results" 

# Load all *_model_performance.csv files
all_files = glob.glob(os.path.join(directory_path, "*_model_performance.csv"))

# Check if any files were found
if not all_files:
    raise FileNotFoundError("No model_performance.csv files found in the specified directory.")

# Read each file and store valid dataframes
df_list = []
for file in all_files:
    try:
        df_temp = pd.read_csv(file)
        df_temp["Gene"] = os.path.basename(file).split("_")[0]  # Extract gene name from filename
        df_list.append(df_temp)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Ensure we have valid data
if not df_list:
    raise ValueError("No valid, non-empty CSV files were found.")

# Concatenate all dataframes into a single dataframe
df = pd.concat(df_list, ignore_index=True)

# Ensure column names are correctly formatted
df.columns = df.columns.str.strip()

# Plotting
plt.figure(figsize=(12, 6))

# Using different colors for each gene
genes = df["Gene"].unique()
colors = plt.get_cmap("tab10")  # Define colormap

for i, gene in enumerate(genes):
    subset = df[df["Gene"] == gene]
    plt.barh(subset["Best Model"], subset["Spearman Correlation"], label=gene, color=colors(i / len(genes)), alpha=0.7)

plt.xlabel("Spearman Correlation")
plt.ylabel("Best Model")
plt.title("Model Performance for Different Genes")
plt.xlim(0, 1)
plt.legend(title="Gene", loc="lower right")
plt.gca().invert_yaxis()

# Show the plot
plt.show()


In [ ]:
# Re-import necessary libraries after execution reset
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the directory containing the CSV files
directory_path = "./esm/gene_results"  # Update with correct path

# Load all *_model_performance.csv files
all_files = glob.glob(os.path.join(directory_path, "*_model_performance.csv"))

# Check if any files were found
if not all_files:
    raise FileNotFoundError("No model_performance.csv files found in the specified directory.")

# Read each file and store valid dataframes
df_list = []
for file in all_files:
    try:
        df_temp = pd.read_csv(file)
        df_temp["Gene"] = os.path.basename(file).split("_")[0]  # Extract gene name from filename
        df_list.append(df_temp)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Ensure we have valid data
if not df_list:
    raise ValueError("No valid, non-empty CSV files were found.")

# Concatenate all dataframes into a single dataframe
df = pd.concat(df_list, ignore_index=True)

# Ensure column names are correctly formatted
df.columns = df.columns.str.strip()

# Extracting only relevant models (KNN, SVR, Random Forest)
df_filtered = df[df["Best Model"].str.contains("KNeighborsRegressor|SVR|RandomForestRegressor", na=False)]

# Mapping specific models to simplified names
df_filtered["Model Type"] = df_filtered["Best Model"].apply(
    lambda x: "KNN" if "KNeighborsRegressor" in x else 
              "SVR" if "SVR" in x else 
              "Random Forest" if "RandomForestRegressor" in x else x
)

# Creating dummy spectral parameter values (replace with actual data if available)
df_filtered["Spectral Parameter"] = np.linspace(0.1, 1, len(df_filtered))

# Creating dummy error bars (variation in correlation values, replace if real std exists)
df_filtered["Error"] = np.random.uniform(0.02, 0.05, len(df_filtered))

# Plotting
plt.figure(figsize=(10, 6))

# Colors for different models
colors = {"KNN": "blue", "SVR": "orange", "Random Forest": "green"}

for model in df_filtered["Model Type"].unique():
    subset = df_filtered[df_filtered["Model Type"] == model]
    plt.errorbar(
        subset["Spectral Parameter"], subset["Spearman Correlation"],
        yerr=subset["Error"], fmt="o", label=model, color=colors[model], capsize=4
    )

plt.axhline(y=0.5, color="red", linestyle="dashed")  # Reference line
plt.xlabel("Spectral Parameter (λ)")
plt.ylabel("Spearman Correlation")
plt.title("Spectral Performance of Models")
plt.legend(title="Model Type", loc="lower left")
plt.ylim(0.2, 1)

# Show the plot
plt.show()


In [ ]:
# Extract unique genes
unique_genes = df_filtered["Gene"].unique()

# Assign numerical values to genes for x-axis
gene_to_x = {gene: i + 1 for i, gene in enumerate(unique_genes)}
df_filtered["Gene X"] = df_filtered["Gene"].map(gene_to_x)

# Plotting
plt.figure(figsize=(12, 6))

# Colors for different models
colors = {"KNN": "blue", "SVR": "orange", "Random Forest": "green"}

for model in df_filtered["Model Type"].unique():
    subset = df_filtered[df_filtered["Model Type"] == model]
    plt.errorbar(
        subset["Gene X"], subset["Spearman Correlation"],
        yerr=subset["Error"], fmt="o", label=model, color=colors[model], capsize=4
    )

plt.axhline(y=0.5, color="red", linestyle="dashed")  # Reference line
plt.xticks(list(gene_to_x.values()), list(gene_to_x.keys()), rotation=45)  # Set gene labels on x-axis
plt.xlabel("Genes")
plt.ylabel("Spearman Correlation")
plt.title("Model Performance for Each Gene: ESM Supervised Variant Prediction Task")
plt.legend(title="Model Type", loc="upper left")
plt.ylim(0.0, 1)

# Show the plot
plt.show()
